In [30]:
# languages='english'
# rewards_in_the_name = 'FR'

# from datasets import Dataset, DatasetDict, load_from_disk
from datasets import load_dataset
import pandas as pd
import evaluate
import json
import re
import numpy as np
# from PIL import Image
# import pytesseract
# from ddgs import DDGS
from qwen_vl_utils import process_vision_info
from tqdm import tqdm

dataset1 = load_dataset('Vikir2411CS19/IPRET-V1',split='train')
# dataset1 = load_dataset('Vikir2411CS19/Final-text-F-V3',split='train')

dataset1 = dataset1.add_column("language_encoded", dataset1["language"])
dataset1 = dataset1.class_encode_column("language_encoded")
dataset1 = dataset1.filter(lambda row: row['language']=='gujarati', num_proc=4)

split_dataset1 = dataset1.train_test_split(test_size=0.3, seed=42, stratify_by_column="language_encoded")

dataset1 = split_dataset1["test"]


In [31]:
import torch
# from transformers import Qwen2_5_VLForConditionalGeneration,AutoProcessor,AutoTokenizer 
from transformers import AutoModelForCausalLM ,AutoProcessor,AutoTokenizer 

checkpoint_path = '../BPR_text-Qwen2.5-3B-mcq_guj'
model_id = "Qwen/Qwen2.5-3B-Instruct"
# processor = AutoProcessor.from_pretrained(checkpoint_path, use_fast=True, padding_side="left")
processor = AutoProcessor.from_pretrained(model_id, use_fast=True, padding_side="left")

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     checkpoint_path,
#     # model_id,
#     dtype=torch.bfloat16,
#     device_map="auto",
#     # use_cache= False
# )

model = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    # model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    # use_cache= False
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [32]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a MCQ question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the correct option. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> correct option here </answer>"
)

In [33]:
def format_helper_userinput(example):
    # def format_helper(example):
    # Language-specific option label mapping
    option_label_map = {
        "english": {
            "a":"a",
            "b":"b",
            "c":"c",
            "d":"d"
        },
        "hindi": {
            "a": "क",
            "b": "ख",
            "c": "ग",
            "d": "घ"
        },
        "bengali": {
            "a": "ক",
            "b": "খ",
            "c": "গ",
            "d": "ঘ"
        },
        "marathi": {
            "a": "क",
            "b": "ख",
            "c": "ग",
            "d": "घ"
        },
        "gujarati": {
            "a": "ક",
            "b": "ખ",
            "c": "ગ",
            "d": "ઘ"
        },
        "tamil": {
            "a": "௧",
            "b": "௨",
            "c": "௩",
            "d": "௪"
        }
    }

    # Normalize language and answer key
    language = example.get("language", "").strip().lower()
    # answer_key = example.get("Final Answer", "").strip().lower()  # a/b/c/d
    native_answer = example.get("final_answer")
    lang_map = option_label_map.get(language, {})
    reverse_map = {v: k for k, v in lang_map.items()}  # native → a/b/c/d

    # Resolve option key
    try:
        option_key_letter = reverse_map.get(native_answer)
        option_text = (
            example.get(f"option_{option_key_letter}")
            if option_key_letter else None
        )
    except:
        option_key_letter = native_answer
        option_text = native_answer

    

    user_input = (
        f"Question: {example['question']}\n"
        f"Options: "
        f"{lang_map.get('a','a')}. {example['option_a']}, "
        f"{lang_map.get('b','b')}. {example['option_b']}, "
        f"{lang_map.get('c','c')}. {example['option_c']}, "
        f"{lang_map.get('d','d')}. {example['option_d']}."
    )

    return {
        "user_input": user_input,
        "native_answer_label": native_answer,
        "option_key": option_key_letter,
        "answer_text": option_text
    }

def format_data1(example):
    # templist = []
    result_format_helper = format_helper_userinput(example)
    user_input = result_format_helper['user_input']
    # if example['final_answer']==None:
    #     sol = f"<think>{example['Reasoning']}</think><answer>None</answer>"
    try:
        sol = (
        f"<think> None </think>"
        f"<answer>{result_format_helper['native_answer_label']}. {result_format_helper['answer_text']}</answer>"
        )
    except:
        sol = f"<think> None </think> <answer> {example['final_answer']} </answer>"
    # returned_dict = [
    #       {
    #           "role": "system",
    #           "content": [
    #               {
    #                   "type": "text",
    #                   "text": SYSTEM_PROMPT
    #               }
    #           ],
    #       },
    #       {
    #           "role": "user",
    #           "content": [
    #               {
    #                   "type": "text",
    #                   "text": user_input,
    #               }
    #           ],
    #       },
    #   ]
    returned_dict = [
          {
              "role": "system",
              "content": SYSTEM_PROMPT,
          },
          {
              "role": "user",
              "content": user_input,
          },
      ]

    return {
        # "images":[],
        "item_id": example['item_id'],
        "question": user_input,
      "prompt": returned_dict,
        "solution":sol,
      }

In [35]:
dataset=[format_data1(example) for example in dataset1]

In [37]:
import torch

@torch.inference_mode()
def batch_infer_from_messages(records, model, processor, batch_size=16, max_new_tokens=1024):
    model.eval()
    all_outputs = []

    for start in tqdm(range(0, len(records), batch_size),desc="Batch inference",unit="batch"):
        batch = records[start:start + batch_size]
        # print('1')
        texts = []
        # image_inputs = []
        for ex in batch:
            msgs = ex["prompt"]
            texts.append(processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

            # imgs, _ = process_vision_info(msgs)   # returns list of images for that sample
            # image_inputs.append(imgs)

        inputs = processor(
            text=texts,
            # images=image_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        outputs = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        all_outputs.extend(outputs)

    return all_outputs


In [38]:
preds = batch_infer_from_messages(dataset, model, processor,64)

Batch inference: 100%|██████████| 30/30 [33:01<00:00, 66.05s/batch]


In [43]:
# import json

rowstw = []
for item, pred in zip(dataset, preds):
    rowstw.append({
        "item_id":json.dumps(item["item_id"], ensure_ascii=False),
        "prompt": json.dumps(item["question"], ensure_ascii=False),
        "solution": item["solution"],
        
        "predictions": pred,
    })


In [44]:
df = pd.DataFrame(rowstw)
df.to_csv("GRPO_results_guj.csv", index=False, encoding="utf-8-sig")
